# Lab 12 - Evaluate retrieval and grounded answers

## What are we evaluating?

Lab 11 traced **what happened** during one request. This lab asks whether the result was useful.

A grounded-answer example has three important fields:

- **Query:** what the user asked.
- **Context:** the passage supplied to the model after retrieval.
- **Response:** the answer produced from that passage.

We score three different parts of that flow:

| Evaluator | What it checks |
|---|---|
| Retrieval | Does the context contain information useful for the query? |
| Groundedness | Are the response's claims supported by the context? |
| Relevance | Does the response address the query? |

These scores can disagree for good reasons. If retrieval returns the wrong passage and the response says *"the supplied context does not answer this question,"* Retrieval can fail while Groundedness passes. If an answer sounds relevant but invents a claim, Relevance can pass while Groundedness fails.

Foundry's built-in evaluators use a model to apply a scoring rubric. Treat the score and reason as evidence, not unquestionable truth; results can vary slightly between runs.

In this lab you will freeze four query-context-response examples, score them, repair one bad response without changing its context, and compare the two runs.

## New words

- **Evaluation** - a reusable definition of the row shape and checks to run.
- **Evaluation run** - one execution of that definition on a dataset.
- **Data mapping** - the link between a dataset field and the evaluator input that should receive it.
- **Regression case** - a known failure kept in the test set so it cannot return unnoticed.

## Before you start

Run `az login`, select Python 3.11+, and use the same Foundry project and model deployment as Lab 11. You need Foundry User access. Cloud evaluation uses model quota and can take several minutes.

Replace each `...` blank before running its cell.

Four workshop rows teach the method; they are not evidence that a medical system is ready for clinical use.

In [ ]:
%pip install -q "azure-ai-projects==2.3.0" "azure-identity==1.25.3" "openai==2.54.0"

## 0. Connect and prepare cloud runs

The next cell signs in, creates the Foundry evaluation client and defines a helper for waiting on evaluation runs.

A cloud run is asynchronous: Foundry starts the job and returns immediately, while scoring continues in Azure. `wait_for_run` checks the status until the job finishes, then waits until all row-level results are available. This prevents the notebook from reading an apparently empty report too early.

The helper also converts SDK result objects into ordinary Python data so later cells can print scores and reasons consistently.

**You should see** `Ready` and a unique suffix used to keep this run's names separate.

In [ ]:
import copy
import os
import sys
import time
from uuid import uuid4

from azure.ai.projects import AIProjectClient
from azure.identity import AzureCliCredential

PROJECT_ENDPOINT = os.getenv("AZURE_AI_PROJECT_ENDPOINT", "")
MODEL_DEPLOYMENT = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME", "")
if sys.version_info < (3, 11):
    raise RuntimeError("Select a Python 3.11 or later kernel.")
if not PROJECT_ENDPOINT or not MODEL_DEPLOYMENT:
    raise ValueError("Set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME.")


def check_todos(**answers: object) -> None:
    still_open = [name for name, value in answers.items() if value is ...]
    if still_open:
        raise ValueError(f"Fill in these blanks first: {', '.join(still_open)}")


def primitive(value):
    if hasattr(value, "model_dump"):
        return value.model_dump(mode="json")
    if hasattr(value, "as_dict"):
        return value.as_dict()
    return value


def wait_for_run(eval_id, run, expected_items, timeout_seconds=1200):
    deadline = time.monotonic() + timeout_seconds
    while run.status not in ("completed", "failed", "canceled"):
        if time.monotonic() > deadline:
            raise TimeoutError("Evaluation exceeded 20 minutes; check quota or cancel it in Foundry.")
        time.sleep(5)
        run = client.evals.runs.retrieve(run_id=run.id, eval_id=eval_id)
        print("status:", run.status)
    items = list(client.evals.runs.output_items.list(run_id=run.id, eval_id=eval_id))
    if run.status != "completed":
        raise RuntimeError(f"Evaluation ended as {run.status}: {primitive(getattr(run, 'error', None))}")
    item_deadline = time.monotonic() + 120
    while len(items) < expected_items:
        if time.monotonic() > item_deadline:
            raise TimeoutError(f"Only {len(items)}/{expected_items} output items became visible.")
        time.sleep(2)
        items = list(client.evals.runs.output_items.list(run_id=run.id, eval_id=eval_id))
    return run, items


SUFFIX = uuid4().hex[:8]
credential = AzureCliCredential()
project = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
client = project.get_openai_client(timeout=300, max_retries=0)
print(f"Ready. Suffix: {SUFFIX}")

## 1. Create a fixed baseline dataset

A fair comparison needs stable input. These examples already contain the query, retrieved context and response, so a later score change cannot be caused by live retrieval returning a different passage.

Each row also has a `case_id` for tracking and a `ground_truth` written by the workshop author as a human reference. The three evaluators in this lab score the query, context and response directly; they do not treat a previous model answer as truth.

The four rows cover distinct situations:

- `Q-01` and `Q-02` are expected to be well retrieved, grounded and relevant.
- `Q-03` makes a causal claim that its context explicitly does not support.
- `Q-04` has irrelevant context, but the response safely says it cannot answer from that context.

This makes `Q-03` a **generation failure** and `Q-04` a **retrieval failure**. Keeping both helps show why one overall score would be misleading.

**You should see** four unique case IDs and exactly one row marked as the known bad answer.

In [ ]:
QUALITY_ROWS = [
    {
        "case_id": "Q-01",
        "query": "At what confirmed blood pressure does WHO usually recommend starting drug treatment for hypertension in adults?",
        "context": (
            "WHO hypertension guideline: initiate pharmacological treatment for a confirmed "
            "diagnosis at systolic blood pressure at least 140 mmHg or diastolic blood pressure "
            "at least 90 mmHg."
        ),
        "response": "For confirmed adult hypertension, WHO uses at least 140 mmHg systolic or 90 mmHg diastolic as the usual treatment threshold.",
        "ground_truth": "A confirmed diagnosis at systolic 140 mmHg or higher, or diastolic 90 mmHg or higher.",
        "expected_pattern": "pass",
    },
    {
        "case_id": "Q-02",
        "query": "How should facility staffing relate to workload for infection prevention and control?",
        "context": (
            "WHO IPC core components: staffing levels should be adequately assigned according "
            "to patient workload, and bed occupancy should not exceed standard capacity."
        ),
        "response": "Staffing should be matched to patient workload, alongside control of bed occupancy within the facility's standard capacity.",
        "ground_truth": "Staffing should be appropriate to patient workload and considered with bed occupancy.",
        "expected_pattern": "pass",
    },
    {
        "case_id": "Q-03",
        "query": "Does Ward 4B's low staffing score prove that staffing caused individual infections?",
        "context": (
            "Workshop data note: the ward score is an invented internal maturity score. It is "
            "not a WHO instrument, patient record, causal study, or diagnosis."
        ),
        "response": "Yes. The score proves that understaffing caused infections in Ward 4B.",
        "ground_truth": "No. The synthetic maturity score cannot establish patient-level outcomes or causation.",
        "expected_pattern": "fail",
    },
    {
        "case_id": "Q-04",
        "query": "What amoxicillin dose should a child with acute otitis media receive?",
        "context": "The retrieved passage describes facility-level infection prevention programme governance.",
        "response": "The supplied context does not cover paediatric antibiotic dosing, so I cannot answer from it.",
        "ground_truth": "Abstain because this knowledge base and context do not cover the requested patient-specific dose.",
        "expected_pattern": "retrieval_fail_but_safe_response",
    },
]

assert len({row["case_id"] for row in QUALITY_ROWS}) == len(QUALITY_ROWS)
assert [row["case_id"] for row in QUALITY_ROWS if row["expected_pattern"] == "fail"] == ["Q-03"]
assert all(all(row[field] for field in ("query", "context", "response", "ground_truth")) for row in QUALITY_ROWS)
print("Cases:", [row["case_id"] for row in QUALITY_ROWS])

### To-Do 1 - Give each evaluator the fields it needs

The evaluators do not guess which dataset columns to read. A **data mapping** connects each evaluator input to a field in the current row.

Use only the evidence needed for each question:

- Retrieval compares `query` with `context`.
- Groundedness compares `response` with both `query` and `context`.
- Relevance compares `response` with `query`.

Set `METRIC_SPECS` to the three built-in evaluator names and mappings. The `{{item.query}}` syntax means “take `query` from this dataset row.”

**Predict:** for `Q-04`, which evaluator should fail because the passage is unrelated, and which evaluators may still approve the safe abstention?

**Key concept:** mapping the wrong field means asking the evaluator the wrong question.

<details><summary>Show solution code</summary>

```python
METRIC_SPECS = {
    "retrieval": ("builtin.retrieval", {"query": "{{item.query}}", "context": "{{item.context}}"}),
    "groundedness": ("builtin.groundedness", {"query": "{{item.query}}", "context": "{{item.context}}", "response": "{{item.response}}"}),
    "relevance": ("builtin.relevance", {"query": "{{item.query}}", "response": "{{item.response}}"}),
}
```

</details>

In [ ]:
METRIC_SPECS = ... # TODO 1: evaluator names and field mappings.
check_todos(METRIC_SPECS=METRIC_SPECS)
assert set(METRIC_SPECS) == {"retrieval", "groundedness", "relevance"}
assert all(name.startswith("builtin.") for name, _ in METRIC_SPECS.values())
print("Configured metric contracts:", list(METRIC_SPECS))

## 2. Create the reusable evaluation

The next cell tells Foundry two things:

1. **What one row looks like.** `DataSourceConfigCustom` defines the allowed fields and required fields, similar to the JSON Schemas used for tools in Lab 4.
2. **Which checks to run.** Each `TestingCriterionAzureAIEvaluator` selects one built-in evaluator, supplies the model deployment used to judge the row, and applies the mapping from the previous section.

The resulting **evaluation** is the reusable test definition. It contains no results yet. Results belong to **runs**, which lets the same checks score a baseline and a repaired dataset without redefining the test.

**You should see** one evaluation ID and the three criterion names: Retrieval, Groundedness and Relevance.

In [ ]:
from azure.ai.projects.models import TestingCriterionAzureAIEvaluator
from openai.types.eval_create_params import DataSourceConfigCustom

data_source_config = DataSourceConfigCustom(
    type="custom",
    item_schema={
        "type": "object",
        "properties": {
            "case_id": {"type": "string"},
            "query": {"type": "string"},
            "context": {"type": "string"},
            "response": {"type": "string"},
            "ground_truth": {"type": "string"},
            "expected_pattern": {"type": "string"},
        },
        "required": ["case_id", "query", "context", "response", "ground_truth"],
        "additionalProperties": False,
    },
    include_sample_schema=True,
)
testing_criteria = [
    TestingCriterionAzureAIEvaluator(
        type="azure_ai_evaluator",
        name=metric_name,
        evaluator_name=evaluator_name,
        initialization_parameters={"deployment_name": MODEL_DEPLOYMENT},
        data_mapping=mapping,
    )
    for metric_name, (evaluator_name, mapping) in METRIC_SPECS.items()
]
evaluation = client.evals.create(
    name=f"day3-grounded-answer-eval-{SUFFIX}",
    data_source_config=data_source_config,
    testing_criteria=testing_criteria,
)
print({"evaluation_id": evaluation.id, "criteria": list(METRIC_SPECS)})

## 3. Score the baseline

This cell creates the first **evaluation run** and sends the four frozen rows directly to Foundry as JSONL data. It then waits for cloud scoring and prints every row's result.

Read the results row by row, not just as an average. For each evaluator, inspect:

- the score or pass label;
- the reason the evaluator gives;
- whether that reason matches the failure you intentionally placed in the row.

Expected pattern:

- `Q-03` should be weak on Groundedness because its causal claim contradicts the context.
- `Q-04` should be weak on Retrieval because the context does not answer the dosing question, even though the response abstains safely.

The exact model-generated scores can vary. The diagnostic pattern and reasons matter more than a particular number.

**You should see** four output items, evaluator results and reasons, a report URL, and a final baseline `PASS` message.

In [ ]:
baseline_run = client.evals.runs.create(
    eval_id=evaluation.id,
    name=f"day3-grounded-baseline-{SUFFIX}",
    metadata={"dataset": "medical-quality-v1", "variant": "baseline"},
    data_source={
        "type": "jsonl",
        "source": {
            "type": "file_content",
            "content": [{"item": row} for row in QUALITY_ROWS],
        },
    },
)
print({"evaluation_id": evaluation.id, "run_id": baseline_run.id})
baseline_run, baseline_items = wait_for_run(evaluation.id, baseline_run, len(QUALITY_ROWS))
print({"status": baseline_run.status, "report_url": getattr(baseline_run, "report_url", None)})


def print_results(items):
    for item in items:
        data = primitive(item)
        source = data.get("datasource_item", {})
        print(f"\n{source.get('case_id', data.get('item_id', 'case'))}")
        for result in data.get("results", []):
            print(
                f"  {result.get('name')}: score={result.get('score')} "
                f"label={result.get('label')} passed={result.get('passed')}"
            )
            if result.get("reason"):
                print(f"    {result['reason']}")


print_results(baseline_items)
assert len(baseline_items) == len(QUALITY_ROWS)
assert all(primitive(item).get("results") for item in baseline_items)
print("\nPASS - every baseline row returned metric results and reasons for inspection.")

### To-Do 2 - Repair the unsupported answer

Now run a controlled experiment. Change only the `Q-03` response while keeping its query and context fixed.

Write a replacement that follows the supplied evidence: the synthetic ward score can identify an improvement area, but it cannot prove patient-level infections or causation.

The code copies the baseline rows before making the change. The original failure remains available as regression evidence, and the repaired copy becomes a second run under the same evaluation.

**Predict:** Groundedness should improve because the answer changed. Should Retrieval change when the context is identical? A small score movement can be judge variability, but it is not evidence that retrieval improved.

**Key concept:** change one variable at a time so you know which component caused the result.

<details><summary>Show solution code</summary>

```python
FIXED_Q03_RESPONSE = (
    "No. Ward 4B's synthetic maturity score identifies an internal improvement area, "
    "but it cannot prove patient-level infections or causation."
)
```

</details>

In [ ]:
FIXED_Q03_RESPONSE = ...  # TODO 2: a grounded answer that rejects the causal claim.
check_todos(FIXED_Q03_RESPONSE=FIXED_Q03_RESPONSE)

fixed_rows = copy.deepcopy(QUALITY_ROWS)
q03 = next(row for row in fixed_rows if row["case_id"] == "Q-03")
q03["response"] = FIXED_Q03_RESPONSE
assert next(row for row in QUALITY_ROWS if row["case_id"] == "Q-03")["response"] != q03["response"]

fixed_run = client.evals.runs.create(
    eval_id=evaluation.id,
    name=f"day3-grounded-fixed-{SUFFIX}",
    metadata={"dataset": "medical-quality-v1", "variant": "q03-grounded-fix"},
    data_source={
        "type": "jsonl",
        "source": {
            "type": "file_content",
            "content": [{"item": row} for row in fixed_rows],
        },
    },
)
fixed_run, fixed_items = wait_for_run(evaluation.id, fixed_run, len(fixed_rows))
print_results(fixed_items)
print({"baseline_report": getattr(baseline_run, "report_url", None)})
print({"fixed_report": getattr(fixed_run, "report_url", None)})

## Verify that the comparison is fair

Evaluator scores come from a model and can vary, so this check does not require one exact score.

Instead, it verifies the experiment design:

- both cloud runs completed;
- both runs contain the same four cases;
- the original `Q-03` row was preserved;
- only the copied response changed while its context stayed fixed;
- every row returned evaluator results.

These are deterministic facts controlled by the notebook. If they pass, differences between the two reports can be interpreted as evidence about the response repair rather than a changed dataset.

**You should see** a `PASS` message confirming two comparable runs.

In [ ]:
assert baseline_run.status == fixed_run.status == "completed"
assert len(baseline_items) == len(fixed_items) == 4
assert fixed_rows[2]["case_id"] == QUALITY_ROWS[2]["case_id"] == "Q-03"
assert fixed_rows[2]["context"] == QUALITY_ROWS[2]["context"]
assert fixed_rows[2]["response"] != QUALITY_ROWS[2]["response"]
assert all(primitive(item).get("results") for item in fixed_items)
print("PASS - two comparable runs scored the frozen baseline and the one-row generation repair.")

## Turn scores into a decision

An evaluation report is useful only when it leads to a specific next step. Write a short release note that answers:

1. Which **row and evaluator** produced the weakest evidence?
2. Does the likely fix belong in retrieval, response generation, policy, or the test data?
3. Should you `ship`, `mitigate`, or `collect more evidence`?

For four workshop rows, `collect more evidence` is normally the honest decision even if the known Groundedness failure improves. Passing a small dataset does not establish performance on unseen medical questions.

## What you learned

- A RAG result has separable retrieval, groundedness and relevance concerns.
- The evaluation defines the row contract and evaluator suite; each run applies it to data.
- Data mappings determine what evidence each evaluator receives.
- Row-level reasons help locate a failure more effectively than one average.
- A controlled repair keeps the context fixed and changes only the unsupported response.
- Model-based evaluators provide scalable judgement, not deterministic truth or clinical certification.

**Check your understanding**

1. `Q-04` has poor Retrieval but good Groundedness. Why is that possible?
2. Groundedness improves after only the response changes. Which component improved?
3. All four rows pass. What evidence is still missing before release?

<details><summary>Compare your answers</summary>

1. The context is irrelevant, but the response stays supported by it by saying the answer is not present.
2. Response-generation behavior improved; retrieval did not change.
3. You still need representative reviewed cases, repeated runs, safety and tool tests, live-path evidence and an approved release threshold.

</details>

Further reading: [cloud evaluation](https://learn.microsoft.com/azure/foundry/observability/how-to/cloud-evaluation), [RAG evaluators](https://learn.microsoft.com/azure/foundry/concepts/evaluation-evaluators/rag-evaluators), and [evaluation results](https://learn.microsoft.com/azure/foundry/observability/how-to/cloud-evaluation-results).

**Expected artifact:** baseline and repaired report URLs plus a release note based on row-level evidence.

**Finish:** the last cell closes local clients. The evaluation and both reports remain in Foundry for comparison.

**Next:** Lab 13 evaluates not only the final answer, but the sequence of tool calls and results that produced it.

In [ ]:
client.close()
project.close()
credential.close()
print("Closed local clients. The evaluation and both run reports remain in Foundry.")